# FPL Predictive Team

Run the projection from a phone. No terminal, no install on your device — everything happens on Google's machine.

**How to use it:** tap the play button on each cell in order. Cell 2 has boxes to fill in; the rest just run.

Colab can reach the official FPL API, so this uses live prices, form and injury flags — unlike the offline snapshot committed in the repository.

## 1. Set up (about a minute)

In [ ]:
!git clone --depth 1 https://github.com/007-braphiz/FPL-Predictive-Team.git repo 2>/dev/null || (cd repo && git pull -q)
%cd /content/repo
!pip install -q -r requirements.txt
print('ready')

## 2. Your details

`entry_id` is the number in your FPL team's web address:
`fantasy.premierleague.com/entry/`**`1234567`**`/event/4`

Leave it as `0` to use the squad already saved in the repository.

In [ ]:
entry_id       = 0     #@param {type:"integer"}
free_transfers = 1     #@param {type:"integer"}
bank           = 0.0   #@param {type:"number"}
horizon        = 3     #@param {type:"integer"}
max_transfers  = 3     #@param {type:"integer"}

from fplpred import data as D

bundle = D.load('live')
gw = bundle.next_gw
print(f'Live data loaded. Next deadline is gameweek {gw}.')

if entry_id:
    # Pulling the squad from the API also brings in each player's selling
    # price, which is what the transfer optimiser actually budgets against --
    # a player who has risen since you bought him sells for less than his
    # market price, and ignoring that overstates what you can afford.
    from fplpred.cli import main
    main(['import-team', '--entry', str(entry_id), '--gw', str(gw),
          '--free-transfers', str(free_transfers)])
else:
    print('Using data/my_team.json as committed.')

## 3. The answer

Line-up, captain, bench order and whether any transfer beats holding.

In [ ]:
from fplpred.cli import main

main(['analyse', '--source', 'live', '--mobile',
      '--horizon', str(horizon),
      '--free-transfers', str(free_transfers),
      '--bank', str(bank),
      '--max-transfers', str(max_transfers)])

## 4. The full working

Every projected point broken down by source — goals, clean sheet, defensive
contribution, bonus — plus the team ratings the fixture adjustments came from.
Worth reading before trusting any single number.

In [ ]:
from IPython.display import Markdown, display
from pathlib import Path

display(Markdown(Path(f'reports/gw{gw}_report.md').read_text()))

## 5. Optional: what a wildcard would look like

The best legal 15 under budget, ignoring your current squad. Useful as a
reference point even when you are not playing the chip — if your team is
close to this, you are in good shape.

In [ ]:
main(['wildcard', '--source', 'live', '--budget', '100.0',
      '--horizon', str(horizon)])

---

### What this cannot see

* Press conferences and predicted line-ups. Rotation beyond the official
  injury flag is not modelled — apply team news yourself on top.
* Anything about *rank*. The model maximises raw points, not position. If you
  are chasing a mini-league, deliberately picking lower-owned players is
  sometimes correct and this will never suggest it.
* Certainty. A projection of 5.0 is an average over many parallel gameweeks;
  a captain returning 2 is a normal outcome, not a broken model.